# 01 EDA & Feature Engineering: College Football Attendance


In [1]:
import sys
print(sys.executable)

/opt/homebrew/opt/python@3.11/bin/python3.11


In [2]:
import sys
!{sys.executable} -m pip install python-dotenv requests pandas


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [3]:
!pip install python-dotenv requests pandas


zsh:1: command not found: pip


In [4]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env in the project root

CFBD_API_KEY = os.getenv("CFBD_API_KEY")

if not CFBD_API_KEY:
    raise RuntimeError(
        "CFBD_API_KEY not found. Copy .env.example to .env and add your key "
        "(get one at https://collegefootballdata.com/key)."
    )


## Fetch data from the CFBD API

Pulling the `/games` endpoint for a given season as the first slice of data,
this is what attendance figures hang off of. Swap `YEAR` / add `week` or
`team` params as the analysis needs more granularity.

In [5]:
# Call the CFBD API using the key loaded from .env
import requests
import pandas as pd

YEARS = [2023, 2024, 2025]

BASE_URL = "https://api.collegefootballdata.com"
HEADERS = {
    "Authorization": f"Bearer {CFBD_API_KEY}",
    "Accept": "application/json",
}

all_games = []
for year in YEARS:
    response = requests.get(
        f"{BASE_URL}/games",
        headers=HEADERS,
        params={"year": year, "seasonType": "regular"},
        timeout=30,
    )
    response.raise_for_status()
    year_games = response.json()
    print(f"Fetched {len(year_games)} rows for {year}.")
    all_games.extend(year_games)

df = pd.DataFrame(all_games)
print(f"Total rows across {len(YEARS)} seasons: {len(df)}")


Fetched 3595 rows for 2023.
Fetched 3747 rows for 2024.
Fetched 3745 rows for 2025.
Total rows across 3 seasons: 11087


## Verify the fetch

In [9]:
#Confirm the fetch worked
df.head(-30)

,id,season,week,seasonType,startDate,startTimeTBD,completed,neutralSite,conferenceGame,attendance,...,awayConference,awayPoints,awayLineScores,awayPostgameWinProbability,awayPregameElo,awayPostgameElo,excitementIndex,highlights,notes,playoff
0,401525434,2023,1,regular,2023-08-26T18:30:00.000Z,False,True,True,False,49000.0,...,American Athletic,3.0,"[0, 0, 0, 3]",0.023594,1471.0,1385.0,1.646466,,NaN,None
1,401540199,2023,1,regular,2023-08-26T19:30:00.000Z,False,True,True,False,NaN,...,UAC,7.0,"[7, 0, 0, 0]",0.031442,NaN,NaN,4.165937,,NaN,None
2,401520145,2023,1,regular,2023-08-26T21:30:00.000Z,False,True,False,True,17982.0,...,Conference USA,14.0,"[0, 7, 0, 7]",0.192517,1369.0,1370.0,4.604490,,NaN,None
3,401525450,2023,1,regular,2023-08-26T23:00:00.000Z,False,True,False,False,15356.0,...,FBS Independents,41.0,"[7, 3, 3, 28]",0.882491,1074.0,1122.0,5.430058,,NaN,None
4,401532392,2023,1,regular,2023-08-26T23:00:00.000Z,False,True,False,False,23867.0,...,Mid-American,13.0,"[3, 3, 0, 7]",0.121683,1482.0,1473.0,5.105937,,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11052,401777327,2025,15,regular,2025-12-06T17:00:00.000Z,False,True,True,False,85519.0,...,Big 12,7.0,"[7, 0, 0, 0]",0.078525,1797.0,1750.0,4.356364,,Big 12 Championship,None
11053,401837095,2025,15,regular,2025-12-06T17:00:00.000Z,False,True,False,False,NaN,...,New Jersey,13.0,"[7, 6, 0, 0]",NaN,NaN,NaN,NaN,,Division III Championship - Third Round,None
11054,401836850,2025,15,regular,2025-12-06T17:00:00.000Z,False,True,False,False,NaN,...,Ohio,10.0,"[0, 0, 7, 0, 0, 3]",NaN,NaN,NaN,NaN,,Division III Championship - Third Round,None
11055,401836218,2025,15,regular,2025-12-06T17:00:00.000Z,False,True,False,False,3042.0,...,MVFC,47.0,"[14, 10, 16, 7]",1.000000,NaN,NaN,2.224629,,FCS Championship - Second Round,None


In [10]:
print("Shape:", df.shape)
df.info()

Shape: (11087, 34)
<class 'pandas.DataFrame'>
RangeIndex: 11087 entries, 0 to 11086
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          11087 non-null  int64  
 1   season                      11087 non-null  int64  
 2   week                        11087 non-null  int64  
 3   seasonType                  11087 non-null  str    
 4   startDate                   11087 non-null  str    
 5   startTimeTBD                11087 non-null  bool   
 6   completed                   11087 non-null  bool   
 7   neutralSite                 11087 non-null  bool   
 8   conferenceGame              11087 non-null  bool   
 9   attendance                  3970 non-null   float64
 10  venueId                     11061 non-null  float64
 11  venue                       11061 non-null  str    
 12  homeId                      11087 non-null  int64  
 13  homeTeam               

## Export to CSV so the team doesn't need their own API key



In [11]:
#Export the fetched data so teammates can work from a static file 

import os

os.makedirs("../data/raw", exist_ok=True)
output_path = f"../data/raw/cfbd_games_{min(YEARS)}_{max(YEARS)}.csv"
df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")


Saved to ../data/raw/cfbd_games_2023_2025.csv
